# **Data Fintering and Querying using pandas**

In [40]:
# Importing required module 
from sqlalchemy import create_engine 
from urllib.parse import quote_plus 
from tabulate import tabulate 
import pandas as pd 
import os 

# Database Creadintial 
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD"))
DB_HOST = "localhost"
DB_PORT = "1433"
DB_NAME = "TestDB"
ODBC_DRIVER = ("ODBC Driver 18 for SQL Server")
TRUST_SERVER_CERTIFICATE = "yes"

# Varify enverment variable
if not DB_USER:
    raise ValueError("DB_USER enverment variable not found")
if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD enverment veriable not found ")

# Database Creadintial for connecting to databse 
connection_string = (
    f"mssql+pyodbc://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    f"?driver={quote_plus(ODBC_DRIVER)}"
    f"&TrustServerCertificate={TRUST_SERVER_CERTIFICATE}"
)

# creating engine for connecting to datbase using connection string 
try : 
    engine = create_engine(connection_string)
    print("Successfully connect to SQL Server databse")

except Exception as error :
    print("Faild to connecting SQL Server Databse")
    raise error 


Successfully connect to SQL Server databse


In [41]:
# querying data for applying filtering logic 
query = """ 
            SELECT 
                * 
            FROM silver.customers ;
        """

In [42]:
# Reating data using pandas dataframe 
try : 
    df = pd.read_sql(query, engine)
    print("Successfully query conplited")

except Exception as error : 
    print("Query faild to datasbe check out the query")
    raise error

Successfully query conplited


#### **Summary of Data quelity check using pandas**

In [63]:
summery  = pd.DataFrame({
    "columns" : df.columns,
    "data_type" : df.dtypes,
    "null_count" : df.isnull().sum(),
    "not_null_count" : df.notnull().sum(),
    "unique_count" : df.nunique(),
    "duplicate_count" : df.apply(lambda col : col.duplicated().sum())
    
})

In [64]:
print(
    tabulate(
        summery,
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌──────────────────────┬──────────────────────┬────────────────┬──────────────┬──────────────────┬────────────────┬───────────────────┐
│                      │ columns              │ data_type      │   null_count │   not_null_count │   unique_count │   duplicate_count │
├──────────────────────┼──────────────────────┼────────────────┼──────────────┼──────────────────┼────────────────┼───────────────────┤
│ customer_id          │ customer_id          │ int64          │            0 │              640 │            640 │                 0 │
├──────────────────────┼──────────────────────┼────────────────┼──────────────┼──────────────────┼────────────────┼───────────────────┤
│ title                │ title                │ object         │          358 │              282 │              5 │               634 │
├──────────────────────┼──────────────────────┼────────────────┼──────────────┼──────────────────┼────────────────┼───────────────────┤
│ first_name           │ first_name           │ 

In [43]:
# breaking dataset using domain logic
customer_identity = ['customer_id', 'title', 'first_name', 'last_name', 'gender', 'is_active']
customer_address = ['customer_id', 'address', 'city', 'state', 'state_abbr', 'zip_code', 'country', 'region']
customer_content = ['customer_id', 'email', 'phone']
customer_business_info = ['customer_id','customer_segment', 'loyalty_points','preferred_channel', 'annual_income_usd', 'company' ]
customer_dates = ['customer_id','date_of_birth', 'account_created_date']

In [44]:
# Creating DataFrame for eatch domain 
try :
    customer_identity_df = df[customer_identity]
    customer_address_df = df[customer_address]
    customer_content_df = df[customer_content]
    customer_business_info_df = df[customer_business_info]
    customer_dates_df = df[customer_dates]

    print("Pandas DataFrame successfully created")

except Exception as erroe : 
    print("Faild to Creating Pandas DataFrame")
    raise error

Pandas DataFrame successfully created


### **Boolean Indexing and filtering data usign pandas** 

In [45]:
# Data Profiling in domain customer_identity DataFrame
print(
    tabulate(
        customer_identity_df.sample(frac=0.01),
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌─────┬───────────────┬─────────┬──────────────┬─────────────┬──────────┬─────────────┐
│     │   customer_id │ title   │ first_name   │ last_name   │ gender   │ is_active   │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│ 512 │          1513 │         │ John         │ Roberts     │ Unknown  │ True        │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│ 324 │          1325 │         │ Brenda       │ Wood        │ Male     │ False       │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│   7 │          1008 │         │ Joyce        │ Wood        │ Female   │ True        │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│ 452 │          1453 │         │ Raymond      │ Foster      │ Male     │ False       │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│ 548 │          1549 │         

#### **Applying boolean masking filter**

In [47]:
customer_identity_df["custoemr_id"] > 300

KeyError: 'custoemr_id'

In [ ]:
mask = df['age'] > 25
df[mask]

# Compound conditions — use & | ~ (not and/or/not)
df[(df['age'] > 25) & (df['score'] >= 80)]
df[(df['city'] == 'NYC') | (df['city'] == 'LA')]
df[~df['name'].str.startswith('A')]

# isin
df[df['status'].isin(['active', 'pending'])]
df[~df['id'].isin(exclude_ids)]

# between (inclusive by default)
df[df['age'].between(25, 35)]
df[df['date'].between('2024-01-01', '2024-12-31')]